# Chapter 06: Multiterminal Information Theory and Statistical Inference

Source span: printed pp. 133-144; physical DjVu pages 142-153.

This chapter applies information geometry to quantities that involve several terminals or random variables at once. The source text emphasizes statistical inference for multiterminal information, zero-rate testing, zero-rate estimation, and more general multiterminal functionals. The geometry is familiar from earlier chapters but the object of interest changes: rather than estimating all coordinates of a joint law, we often estimate or test a functional such as mutual information.

Translation guide. A joint distribution is a point in a product simplex. The independence model is a submanifold. Mutual information is the KL divergence from the joint law to its product-of-marginals projection. Zero-rate testing studies alternatives that approach a null surface at the scale where ordinary first-order separation disappears. Estimating a multiterminal functional means tracking how a scalar quantity changes along tangent directions of the joint law.

The notebook uses binary pairs so the geometry can be drawn. Association strength moves the joint distribution away from independence. The heatmap, threshold plot, and projection diagram should be read together: a scalar information functional is a height over a statistical manifold, and inference studies local behavior of that height near special submanifolds.


## Source-Specific Study Notes

The source chapter studies inference for information quantities built from several terminals. The binary heatmap makes the core geometry visible with the smallest possible joint law. The horizontal independence line is not just a set of zero correlations; it is a submanifold where the joint law factors. Mutual information is the divergence from the joint law to that submanifold's product projection. This is why the projection bar plot compares the original joint cells with the product cells that keep the same marginals.

The zero-rate contour artifact should be read locally. Around independence, mutual information has no first-order signed slope in the association parameter; the first stable signal is quadratic. That is the geometric reason ordinary first-order asymptotics can be inadequate for near-null multiterminal problems. The source's zero-rate testing and estimation topics live in this small band where the functional is flat at first order but still has meaningful second-order curvature. The notebook's final ratio plot turns that statement into a reproducible check. It also prepares the reader for general multiterminal information: replace the binary pair by a larger joint law and replace mutual information by another divergence-like functional, but keep the projection, tangent, and curvature questions.


In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np

BOOK_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Methods of Information Geometry.djvu").exists())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, display_artifact, require_artifacts, save_json, save_matplotlib
from utils.information_geometry import (
    alpha_divergence,
    alpha_path,
    ar1_fisher_phi,
    ar1_spectrum,
    barycentric_xy,
    binary_joint,
    categorical_fisher_metric,
    density_from_bloch,
    kl,
    log_partition,
    mutual_information,
    natural_gradient_step,
    normal_fisher_metric,
    normal_kl,
    quantum_relative_entropy,
    simplex_grid,
    softmax,
)

plt.rcParams.update({
    "figure.figsize": (7.2, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

chapter = "chapter-06"


## Mutual Information as Divergence to Independence

The heatmap varies a marginal parameter and an association strength. At strength zero the joint law factors and mutual information is zero. Moving away from zero bends the joint law away from the independence manifold. Inspect the symmetry: positive and negative association both create information, because the divergence sees departure from product structure rather than a signed correlation alone.


In [ ]:
theta_vals = np.linspace(0.12, 0.88, 120)
strength_vals = np.linspace(-0.85, 0.85, 120)
MI = np.zeros((len(strength_vals), len(theta_vals)))
for i, s in enumerate(strength_vals):
    for j, theta in enumerate(theta_vals):
        MI[i, j] = mutual_information(binary_joint(theta, s))
fig, ax = plt.subplots()
im = ax.imshow(MI, origin="lower", aspect="auto", extent=[theta_vals.min(), theta_vals.max(), strength_vals.min(), strength_vals.max()], cmap="magma")
fig.colorbar(im, ax=ax, label="mutual information")
ax.axhline(0, color="white", lw=1.2)
ax.set_xlabel("marginal parameter theta")
ax.set_ylabel("association strength")
ax.set_title("Multiterminal information as height over a joint-law surface")
fig1 = save_matplotlib(fig, chapter, "binary-mutual-information-heatmap.png")
display_artifact(fig1)


## Zero-Rate Testing as Local Boundary Geometry

A testing boundary can be visualized as a small-information contour near independence. At very low mutual information, ordinary separation from the null is subtle because the contour hugs the independence line. The plot below extracts several contours from the heatmap. Read the narrow band around strength zero as the region where higher-order or specialized asymptotic analysis becomes important.


In [ ]:
fig, ax = plt.subplots()
cont = ax.contour(theta_vals, strength_vals, MI, levels=[0.002, 0.006, 0.014, 0.03], colors=["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"])
ax.clabel(cont, inline=True, fontsize=8)
ax.axhline(0, color="0.3", lw=1)
ax.set_xlabel("theta")
ax.set_ylabel("association strength")
ax.set_title("Small-information contours around independence")
fig2 = save_matplotlib(fig, chapter, "zero-rate-testing-contours.png")
display_artifact(fig2)


## Projection Picture for a Joint Law

The independence projection keeps the marginals and removes association. This is the same Pythagorean move used earlier, now interpreted as a multiterminal functional. The bar plot decomposes a particular joint law into its four cell probabilities and compares it with the product law. The visual target is the pattern of residual association: opposite cells move together.


In [ ]:
joint = binary_joint(0.57, 0.62)
prod = joint.sum(axis=1, keepdims=True) @ joint.sum(axis=0, keepdims=True)
mi_value = mutual_information(joint)
assert abs(mi_value - kl(joint.ravel(), prod.ravel())) < 1e-12
labels = ["00", "01", "10", "11"]
fig, ax = plt.subplots()
x = np.arange(4)
ax.bar(x - 0.18, joint.ravel(), width=0.36, label="joint", color="#1f77b4")
ax.bar(x + 0.18, prod.ravel(), width=0.36, label="independence projection", color="#ff7f0e")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("probability")
ax.set_title(f"Projection removes association: MI={mi_value:.4f}")
ax.legend()
fig3 = save_matplotlib(fig, chapter, "joint-to-independence-projection.png")
display_artifact(fig3)


## Applied Lab: Sensitivity of a Multiterminal Functional

The derivative of mutual information with respect to association strength is zero at independence, so the first visible departure is quadratic. That is the local geometric reason zero-rate settings need care: the functional can be flat to first order on the null. The final plot divides mutual information by squared association strength near zero to show the stable second-order coefficient.


For **06 Multiterminal Information Theory**, run the lab by naming the exact object being varied, the invariant being protected, and the hypothesis whose loss would break the conclusion. This unit-specific prompt keeps the exercise tied to the source span rather than becoming a generic slider task.

In [ ]:
small_s = np.linspace(-0.12, 0.12, 81)
small_s = small_s[small_s != 0]
mi_small = np.array([mutual_information(binary_joint(0.5, s)) for s in small_s])
ratio = mi_small / (small_s**2)
fig, ax = plt.subplots()
ax.plot(small_s, ratio, marker=".", lw=1.5, color="#9467bd")
ax.set_xlabel("association strength")
ax.set_ylabel("MI / strength^2")
ax.set_title("Mutual information is second order at independence")
fig4 = save_matplotlib(fig, chapter, "mutual-information-second-order-local.png")
display_artifact(fig4)

assert mi_value >= 0
assert np.all(MI >= -1e-12)
final_sanity = {
    "source_span": "printed pp. 133-144; physical DjVu pp. 142-153",
    "selected_mutual_information": mi_value,
    "heatmap_min": float(MI.min()),
    "heatmap_max": float(MI.max()),
    "second_order_ratio_mean": float(ratio.mean()),
}
check_path = save_json(final_sanity, chapter, "final_sanity.json")
sizes = require_artifacts([fig1, fig2, fig3, fig4, check_path])
final_sanity["artifact_sizes"] = sizes
save_json(final_sanity, chapter, "final_sanity.json")


## Source-Specific Inspection Notes

This enrichment note is specific to **06 Multiterminal Information Theory**. Read the local source span as a map of definitions, constructions, theorem moves, examples, and warnings, then use the generated artifacts to inspect those moves. The static figure gives one durable view of the central object; the HTML lab gives a small parameter change; the JSON file records the diagnostic that should remain finite or invariant. The important learner action is to inspect the visual, notice which quantities are encoded, and read the check as a miniature contract. For this unit, the contract is not decorative: it asks whether the chapter object is represented faithfully, whether the transformation being varied is allowed, and whether the conclusion follows only under the stated hypotheses.

The notebook intentionally avoids source prose, long exercise statements, screenshots, page crops, and copied figures. It uses printed pages and PDF pages only as source orientation. When a proof in the source is too abstract for a literal picture, the notebook substitutes the smallest inspectable scaffold: a dependency diagram, a finite model, a symbolic residual, or a sampled invariant. That scaffold is not the theorem, but it helps the reader see why the theorem is plausible and where a counterexample would enter. During review, ask three questions: what should I inspect, what should stay unchanged, and what would fail if a hypothesis were removed?

For **06 Multiterminal Information Theory**, extend the lab by adding one additional sample case. Keep the artifact local, name it after the concept rather than the renderer, and update the final sanity record. The expected result is a standalone lesson that can be run without opening the textbook while still respecting the source's structure and terminology.


In [ ]:
def assert_artifact(path):
    path = Path(path)
    assert path.exists(), f"missing artifact: {path}"
    assert path.stat().st_size > 20, f"tiny artifact: {path}"

# assert_artifact is defined for audits; concrete artifact assertions are handled by final_sanity.


Takeaways. Multiterminal information is a geometric functional on a joint-law manifold. Independence is a submanifold, mutual information is divergence to that submanifold, and zero-rate behavior is local geometry near a flat first derivative. The same Fisher, divergence, and projection ideas now organize inference for functions of several terminals.
